In [1]:
import sys
sys.path.append(".")
import numpy as np
from scipy import ndimage
from skimage.draw import ellipsoid
from skimage.morphology import ball
from vqa_utils import compute_shape_descriptors

In [2]:
def make_sphere(dim=100, r=15, center=None):
    if center is None:
        center = (dim//2, dim//2, dim//2)
    zz, yy, xx = np.ogrid[:dim, :dim, :dim]
    mask = (zz-center[0])**2 + (yy-center[1])**2 + (xx-center[2])**2 <= r**2
    return mask

def make_ellipsoid(dim=120, axes=(30,10,10)):
    e = ellipsoid(*axes)          # returns bool array
    mask = np.zeros((dim,dim,dim), bool)
    z0,y0,x0 = e.shape
    start = [(dim-s)//2 for s in e.shape]
    mask[start[0]:start[0]+z0,
         start[1]:start[1]+y0,
         start[2]:start[2]+x0] = e
    return mask

def make_flat_disk(dim=120, radius=30, thickness=3):
    mask = np.zeros((dim,dim,dim), bool)
    zz, yy, xx = np.ogrid[:dim, :dim, :dim]
    cyl = ((yy-dim//2)**2 + (xx-dim//2)**2 <= radius**2) & \
          (np.abs(zz-dim//2) <= thickness)
    return cyl

def make_dumbbell(dim=120, r=20, neck=5, half_len=30):
    """
    Two equal spheres connected by a narrow cylindrical bridge.

    Parameters
    ----------
    dim : int
        Size of cubic volume (dim × dim × dim).
    r : int
        Radius of each end‑sphere.
    neck : int
        Radius of the cylindrical neck.
    half_len : int
        Half‑distance between sphere centres along x‑axis.
    """
    mask = np.zeros((dim, dim, dim), dtype=bool)
    zc, yc = dim // 2, dim // 2
    left_x  = dim // 2 - half_len
    right_x = dim // 2 + half_len

    zz, yy, xx = np.ogrid[:dim, :dim, :dim]

    # left & right spheres
    mask |= (zz - zc) ** 2 + (yy - yc) ** 2 + (xx - left_x)  ** 2 <= r ** 2
    mask |= (zz - zc) ** 2 + (yy - yc) ** 2 + (xx - right_x) ** 2 <= r ** 2

    # cylindrical neck between the two sphere centres
    neck_cylinder = ((zz - zc) ** 2 + (yy - yc) ** 2 <= neck ** 2) & \
                    (xx >= left_x) & (xx <= right_x)
    mask |= neck_cylinder
    return mask

def make_scattered(dim=120, n=6, r=5):
    mask = np.zeros((dim,dim,dim), bool)
    np.random.seed(0)
    for _ in range(n):
        cx, cy, cz = np.random.randint(r, dim-r, size=3)
        mask |= make_sphere(dim, r, (cz,cy,cx))
    return mask

def make_big_core_plus_small_sat(dim=120, core_r=25, sat_r=5, n_sat=4, gap=35):
    """one big sphere + several tiny satellites"""
    mask = np.zeros((dim, dim, dim), bool)
    core_center = (dim//2, dim//2, dim//2)
    # big core
    zz, yy, xx = np.ogrid[:dim, :dim, :dim]
    mask |= (zz-core_center[0])**2 + (yy-core_center[1])**2 + (xx-core_center[2])**2 <= core_r**2
    # satellites
    np.random.seed(1)
    for _ in range(n_sat):
        dz, dy, dx = np.random.randint(-gap, gap+1, size=3)
        c = (core_center[0]+dz, core_center[1]+dy, core_center[2]+dx)
        mask |= (zz-c[0])**2 + (yy-c[1])**2 + (xx-c[2])**2 <= sat_r**2
    return mask

def make_three_equal_spheres(dim=120, r=12, offsets=((-30,0,0),(0,30,0),(30,0,0))):
    """3 similar‑sized foci"""
    mask = np.zeros((dim, dim, dim), bool)
    zz, yy, xx = np.ogrid[:dim, :dim, :dim]
    for off in offsets:
        c = (dim//2+off[0], dim//2+off[1], dim//2+off[2])
        mask |= (zz-c[0])**2 + (yy-c[1])**2 + (xx-c[2])**2 <= r**2
    return mask

In [3]:
tests = {
    # Sphericity / roundness
    "sphere_round": make_sphere(),
    "ellipsoid_oval": make_ellipsoid(axes=(35,20,10)),
    "dumbbell_irregular": make_dumbbell(),

    # Elongation / flatness
    "cigar_elongated": make_ellipsoid(axes=(45,8,8)),
    "flat_disk": make_flat_disk(),

    # Solidity (concavity) – dumbbell has low solidity
    "two_touching": make_dumbbell(),

    # Multiplicity / scattered
    "scattered_foci": make_scattered(n=8, r=4),
    "few_foci": make_scattered(n=3, r=4),
    "core_plus_sat": make_big_core_plus_small_sat(),
    "pure_scattered": make_three_equal_spheres(),
    "dumbbell": make_dumbbell()
}

In [4]:
for name, mask in tests.items():
    desc = compute_shape_descriptors(mask)
    print(f"\n{name.upper():<20}")
    print(f"  volume(mm³)      : {desc['volume_mm3']:.0f}")
    print(f"  sphericity       : {desc['sphericity']:.3f}")
    print(f"  elongation       : {desc['elongation']:.3f}")
    print(f"  flatness         : {desc['flatness']:.3f}")
    print(f"  solidity         : {desc['solidity']:.3f}")
    print(f"  compactness      : {desc['compactness']:.3f}")
    print(f"  multiplicity     : {desc['multiplicity']}")
    print(f"  core_fraction  : {desc['core_fraction']:.2f}")
    print(f"  shape_interp     : {desc['shape_interp']}")
    print(f"  satellites_interp: {desc['satellite_interp']}")


SPHERE_ROUND        
  volume(mm³)      : 14147
  sphericity       : 0.918
  elongation       : 1.000
  flatness         : 1.000
  solidity         : 1.000
  compactness      : 0.218
  multiplicity     : 1
  core_fraction  : 1.00
  shape_interp     : round
  satellites_interp: single lesion

ELLIPSOID_OVAL      
  volume(mm³)      : 29235
  sphericity       : 0.754
  elongation       : 1.751
  flatness         : 0.497
  solidity         : 1.000
  compactness      : 0.208
  multiplicity     : 1
  core_fraction  : 1.00
  shape_interp     : oval
  satellites_interp: single lesion

DUMBBELL_IRREGULAR  
  volume(mm³)      : 68501
  sphericity       : 0.706
  elongation       : 3.502
  flatness         : 1.000
  solidity         : 1.000
  compactness      : 0.168
  multiplicity     : 1
  core_fraction  : 1.00
  shape_interp     : elongated
  satellites_interp: single lesion

CIGAR_ELONGATED     
  volume(mm³)      : 11975
  sphericity       : 0.656
  elongation       : 5.702
  flatness     